In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
dbutils.widgets.text("focus_vendor_name", "JAMES CONSTRUCTION GROUP, LLC")

catalog = dbutils.widgets.get("catalog").strip()
focus_vendor_name = dbutils.widgets.get("focus_vendor_name").strip()

if not focus_vendor_name:
    raise ValueError("focus_vendor_name is empty.")

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "325_build_gold_fact_project_bid_role_comparison.py"
RUN_ID = str(uuid.uuid4())

ROLE_CONTEXT_TABLE = f"{catalog}.gold.fact_project_bid_role_context"
ITEM_COMPARISON_TABLE = f"{catalog}.gold.fact_project_item_comparison"
TARGET_TABLE = f"{catalog}.gold.fact_project_bid_role_comparison"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.fact_project_bid_role_comparison
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS
    WITH base AS (
        SELECT *
        FROM {ROLE_CONTEXT_TABLE}
    )

    , item_cmp AS (
        SELECT
              project_id
            , COUNT(*) AS item_comparison_row_count

            , SUM(CASE WHEN focus_amount IS NOT NULL THEN 1 ELSE 0 END) AS focus_item_count
            , SUM(CASE WHEN benchmark_amount IS NOT NULL THEN 1 ELSE 0 END) AS benchmark_item_count
            , SUM(CASE WHEN estimate_amount IS NOT NULL THEN 1 ELSE 0 END) AS estimate_item_count

            , SUM(CASE WHEN focus_amount IS NOT NULL AND benchmark_amount IS NOT NULL THEN 1 ELSE 0 END) AS matched_focus_benchmark_item_count
            , SUM(CASE WHEN focus_amount IS NOT NULL AND estimate_amount IS NOT NULL THEN 1 ELSE 0 END) AS matched_focus_estimate_item_count

            , SUM(COALESCE(focus_amount, 0)) AS focus_item_amount_total
            , SUM(COALESCE(benchmark_amount, 0)) AS benchmark_item_amount_total
            , SUM(COALESCE(estimate_amount, 0)) AS estimate_item_amount_total

            , SUM(COALESCE(focus_vs_benchmark_amount, 0)) AS focus_vs_benchmark_amount_sum
            , AVG(focus_vs_benchmark_amount) AS focus_vs_benchmark_amount_avg
            , SUM(COALESCE(abs_diff, 0)) AS focus_vs_benchmark_abs_amount_sum
            , AVG(abs_diff) AS focus_vs_benchmark_abs_amount_avg
            , AVG(focus_vs_benchmark_ratio) AS focus_vs_benchmark_ratio_avg

            , SUM(CASE WHEN over_under_flag = 'Overbid' THEN 1 ELSE 0 END) AS overbid_item_count
            , SUM(CASE WHEN over_under_flag = 'Underbid' THEN 1 ELSE 0 END) AS underbid_item_count
            , SUM(CASE WHEN over_under_flag = 'Tie' THEN 1 ELSE 0 END) AS tie_item_count

            , SUM(COALESCE(focus_vs_estimate_amount, 0)) AS focus_vs_estimate_amount_sum
            , AVG(focus_vs_estimate_amount) AS focus_vs_estimate_amount_avg

        FROM {ITEM_COMPARISON_TABLE}
        GROUP BY project_id
    )

    , with_item_rollups AS (
        SELECT
              b.*

            , COALESCE(ic.item_comparison_row_count, 0) AS item_comparison_row_count

            , COALESCE(ic.focus_item_count, 0) AS focus_item_count
            , COALESCE(ic.benchmark_item_count, 0) AS benchmark_item_count
            , COALESCE(ic.estimate_item_count, 0) AS estimate_item_count

            , COALESCE(ic.matched_focus_benchmark_item_count, 0) AS matched_focus_benchmark_item_count
            , COALESCE(ic.matched_focus_estimate_item_count, 0) AS matched_focus_estimate_item_count

            , COALESCE(ic.focus_item_amount_total, 0) AS focus_item_amount_total
            , COALESCE(ic.benchmark_item_amount_total, 0) AS benchmark_item_amount_total
            , COALESCE(ic.estimate_item_amount_total, 0) AS estimate_item_amount_total

            , COALESCE(ic.focus_vs_benchmark_amount_sum, 0) AS focus_vs_benchmark_amount_sum
            , ic.focus_vs_benchmark_amount_avg
            , COALESCE(ic.focus_vs_benchmark_abs_amount_sum, 0) AS focus_vs_benchmark_abs_amount_sum
            , ic.focus_vs_benchmark_abs_amount_avg
            , ic.focus_vs_benchmark_ratio_avg

            , COALESCE(ic.overbid_item_count, 0) AS overbid_item_count
            , COALESCE(ic.underbid_item_count, 0) AS underbid_item_count
            , COALESCE(ic.tie_item_count, 0) AS tie_item_count

            , COALESCE(ic.focus_vs_estimate_amount_sum, 0) AS focus_vs_estimate_amount_sum
            , ic.focus_vs_estimate_amount_avg

            , CASE
                  WHEN COALESCE(ic.matched_focus_benchmark_item_count, 0) > 0
                  THEN CAST(ic.overbid_item_count AS DECIMAL(38,10)) / ic.matched_focus_benchmark_item_count
                  ELSE NULL
              END AS overbid_item_pct

            , CASE
                  WHEN COALESCE(ic.matched_focus_benchmark_item_count, 0) > 0
                  THEN CAST(ic.underbid_item_count AS DECIMAL(38,10)) / ic.matched_focus_benchmark_item_count
                  ELSE NULL
              END AS underbid_item_pct

            , CASE
                  WHEN COALESCE(ic.matched_focus_benchmark_item_count, 0) > 0
                  THEN CAST(ic.tie_item_count AS DECIMAL(38,10)) / ic.matched_focus_benchmark_item_count
                  ELSE NULL
              END AS tie_item_pct

            , CASE
                  WHEN b.focus_bid_total_amount IS NOT NULL
                   AND b.focus_bid_total_amount <> 0
                  THEN COALESCE(ic.focus_item_amount_total, 0) / b.focus_bid_total_amount
                  ELSE NULL
              END AS focus_item_rollup_vs_focus_bid_total_ratio

            , CASE
                  WHEN b.benchmark_bid_total_amount IS NOT NULL
                   AND b.benchmark_bid_total_amount <> 0
                  THEN COALESCE(ic.benchmark_item_amount_total, 0) / b.benchmark_bid_total_amount
                  ELSE NULL
              END AS benchmark_item_rollup_vs_benchmark_bid_total_ratio

            , CASE
                  WHEN b.estimate_project_total IS NOT NULL
                   AND b.estimate_project_total <> 0
                  THEN COALESCE(ic.estimate_item_amount_total, 0) / b.estimate_project_total
                  ELSE NULL
              END AS estimate_item_rollup_vs_estimate_total_ratio

        FROM base b
        LEFT JOIN item_cmp ic
            ON b.project_id = ic.project_id
    )

    , final AS (
        SELECT
              b.*

            , CASE
                  WHEN EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.county = b.county
                        AND hx.project_actual_let_date < b.project_actual_let_date
                  )
                  THEN TRUE ELSE FALSE
              END AS has_focus_vendor_county_history_flag

            , CASE
                  WHEN EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                  )
                  THEN TRUE ELSE FALSE
              END AS has_focus_vendor_project_classification_history_flag

            , (
                SELECT MIN(hx.focus_bid_total_amount)
                FROM with_item_rollups hx
                WHERE hx.focus_vendor_present_flag = TRUE
                  AND hx.project_classification = b.project_classification
                  AND hx.project_actual_let_date < b.project_actual_let_date
                  AND hx.focus_bid_total_amount IS NOT NULL
            ) AS historical_low_focus_bid_for_project_classification

            , (
                SELECT MIN(hx.focus_bid_total_amount)
                FROM with_item_rollups hx
                WHERE hx.focus_vendor_present_flag = TRUE
                  AND hx.county = b.county
                  AND hx.project_classification = b.project_classification
                  AND hx.project_actual_let_date < b.project_actual_let_date
                  AND hx.focus_bid_total_amount IS NOT NULL
            ) AS historical_low_focus_bid_for_county_project_classification

            , CASE
                  WHEN EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.county = b.county
                        AND hx.project_actual_let_date < b.project_actual_let_date
                  )
                   AND EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                  )
                   AND (
                      SELECT MIN(hx.focus_bid_total_amount)
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                        AND hx.focus_bid_total_amount IS NOT NULL
                   ) IS NOT NULL
                   AND b.lowest_bid_amount_in_project IS NOT NULL
                   AND b.lowest_bid_amount_in_project >
                       (
                           SELECT MIN(hx.focus_bid_total_amount)
                           FROM with_item_rollups hx
                           WHERE hx.focus_vendor_present_flag = TRUE
                             AND hx.project_classification = b.project_classification
                             AND hx.project_actual_let_date < b.project_actual_let_date
                             AND hx.focus_bid_total_amount IS NOT NULL
                       )
                  THEN TRUE ELSE FALSE
              END AS is_project_opportunity_flag

            , CASE
                  WHEN b.focus_vendor_present_flag = TRUE
                   AND EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.county = b.county
                        AND hx.project_actual_let_date < b.project_actual_let_date
                   )
                   AND EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                   )
                   AND (
                      SELECT MIN(hx.focus_bid_total_amount)
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                        AND hx.focus_bid_total_amount IS NOT NULL
                   ) IS NOT NULL
                   AND b.lowest_bid_amount_in_project IS NOT NULL
                   AND b.lowest_bid_amount_in_project >
                       (
                           SELECT MIN(hx.focus_bid_total_amount)
                           FROM with_item_rollups hx
                           WHERE hx.focus_vendor_present_flag = TRUE
                             AND hx.project_classification = b.project_classification
                             AND hx.project_actual_let_date < b.project_actual_let_date
                             AND hx.focus_bid_total_amount IS NOT NULL
                       )
                  THEN TRUE ELSE FALSE
              END AS focus_vendor_participated_opportunity_flag

            , CASE
                  WHEN b.focus_vendor_present_flag = FALSE
                   AND EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.county = b.county
                        AND hx.project_actual_let_date < b.project_actual_let_date
                   )
                   AND EXISTS (
                      SELECT 1
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                   )
                   AND (
                      SELECT MIN(hx.focus_bid_total_amount)
                      FROM with_item_rollups hx
                      WHERE hx.focus_vendor_present_flag = TRUE
                        AND hx.project_classification = b.project_classification
                        AND hx.project_actual_let_date < b.project_actual_let_date
                        AND hx.focus_bid_total_amount IS NOT NULL
                   ) IS NOT NULL
                   AND b.lowest_bid_amount_in_project IS NOT NULL
                   AND b.lowest_bid_amount_in_project >
                       (
                           SELECT MIN(hx.focus_bid_total_amount)
                           FROM with_item_rollups hx
                           WHERE hx.focus_vendor_present_flag = TRUE
                             AND hx.project_classification = b.project_classification
                             AND hx.project_actual_let_date < b.project_actual_let_date
                             AND hx.focus_bid_total_amount IS NOT NULL
                       )
                  THEN TRUE ELSE FALSE
              END AS focus_vendor_missed_opportunity_flag

        FROM with_item_rollups b
    )

    SELECT *
    FROM final
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    summary = spark.sql(f"""
        SELECT
              COUNT(*) AS total_rows
            , COUNT(CASE WHEN focus_vendor_present_flag THEN 1 END) AS focus_vendor_present_rows
            , COUNT(CASE WHEN focus_vendor_won_flag THEN 1 END) AS focus_vendor_won_rows
            , COUNT(CASE WHEN is_project_opportunity_flag THEN 1 END) AS opportunity_rows
            , COUNT(CASE WHEN focus_vendor_missed_opportunity_flag THEN 1 END) AS missed_opportunity_rows
        FROM {TARGET_TABLE}
    """).collect()[0]

    log_run(
        "SUCCESS",
        row_count,
        f"Built gold.fact_project_bid_role_comparison successfully for focus vendor {focus_vendor_name}."
    )

    print("=" * 70)
    print("Built gold.fact_project_bid_role_comparison")
    print(f"Focus Vendor: {focus_vendor_name}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Summary:")
    print(f"  total_rows: {summary['total_rows']:,}")
    print(f"  focus_vendor_present_rows: {summary['focus_vendor_present_rows']:,}")
    print(f"  focus_vendor_won_rows: {summary['focus_vendor_won_rows']:,}")
    print(f"  opportunity_rows: {summary['opportunity_rows']:,}")
    print(f"  missed_opportunity_rows: {summary['missed_opportunity_rows']:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise